# Pillar 3: Unstructured Narrative Analyzer
This notebook implements a from-scratch TF-IDF and Multinomial Naive Bayes classifier.
- Uses **CuPy** for GPU acceleration if available.
- Features an automated file finder to avoid path errors on Kaggle/Colab/Local.
- Includes a highly robust Data Intersection logic (with fallbacks for misaligned or dummy datasets).
- Includes a hyperparameter tuning and visualization section to prevent overfitting by checking Validation Loss and Val F1 across different smoothing alphas.


In [ ]:
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt
import os
from IPython.display import FileLink

In [ ]:
# ---------------------------------------------------------
# 1. Data Integration Task & Unbreakable Map Logic
# ---------------------------------------------------------
target_csv = 'insurance_claims.csv'
target_narratives = 'synthetic_auto_narratives.csv'

csv_path = None
narratives_path = None

search_dirs = ['/kaggle/input', '/content', '.', '../input', '/workspace']

for s_dir in search_dirs:
    if os.path.exists(s_dir):
        for root, dirs, files in os.walk(s_dir):
            if target_csv in files:
                csv_path = os.path.join(root, target_csv)
            if target_narratives in files:
                narratives_path = os.path.join(root, target_narratives)

if not csv_path:
    raise FileNotFoundError(f"Could not find {target_csv}. Please check if the dataset is correctly attached!")
if not narratives_path:
    print(f"WARNING: Could not find {target_narratives}. The merge will fall back to physical synthetic limits.")

print(f"Found Main CSV at: {csv_path}")
if narratives_path:
    print(f"Found Narratives CSV at: {narratives_path}\n")

df_main = pd.read_csv(csv_path)
if narratives_path:
    df_narratives = pd.read_csv(narratives_path)
else:
    df_narratives = pd.DataFrame(columns=['policy_number'])

df_main['policy_number_clean'] = df_main['policy_number'].astype(str).str.replace(r'\\.0$', '', regex=True).str.lower().str.strip()
main_ids = set(df_main['policy_number_clean'])

best_col = None
max_overlap = 0

for col in df_narratives.columns:
    narratives_vals = set(df_narratives[col].astype(str).str.replace(r'\\.0$', '', regex=True).str.lower().str.strip())
    overlap = len(main_ids.intersection(narratives_vals))
    if overlap > max_overlap:
        max_overlap = overlap
        best_col = col

EXCEL_TEXT_COLUMN_NAME = ""

if max_overlap > 0:
    print(f"[PK Map] Automatically binding CSV to Narratives via column '{best_col}' ({max_overlap} overlapping IDs detected).")
    df_narratives['policy_number_clean'] = df_narratives[best_col].astype(str).str.replace(r'\\.0$', '', regex=True).str.lower().str.strip()
    df = pd.merge(df_main, df_narratives, on='policy_number_clean', how='inner')
    EXCEL_TEXT_COLUMN_NAME = [c for c in df_narratives.columns if c not in [best_col, 'policy_number_clean']][-1]

elif len(df_main) == len(df_narratives):
    print("[WARN] Zero Policy IDs matched between datasets, BUT ROW COUNTS ARE IDENTICAL!")
    print("[PK Map] Bypassing ID match completely and blindly concatenating columns 1-to-1 based on index alignment.")
    df = pd.concat([df_main.reset_index(drop=True), df_narratives.reset_index(drop=True)], axis=1)
    if len(df_narratives.select_dtypes(include=['object']).columns) > 0:
        EXCEL_TEXT_COLUMN_NAME = df_narratives.select_dtypes(include=['object']).columns[-1]
    else:
        EXCEL_TEXT_COLUMN_NAME = df_narratives.columns[-1]

else:
    print("\n========================== FATAL MERGE ERROR ==========================")
    print("Failed to merge the two files! They contain totally different Policy IDs")
    print(f"and their lengths do not match (CSV: {len(df_main)} rows vs EXCEL: {len(df_narratives)} rows).")
    print("Training Notebook will proceed by synthesizing a dummy narrative on the CSV")
    print("to ensure testing execution is not blocked.")
    print("=======================================================================\n")
    
    df = df_main.copy()
    df['synthetic_narrative'] = "The incident occurred precisely as documented in the police logs. "
    EXCEL_TEXT_COLUMN_NAME = 'synthetic_narrative'

print(f"[Feature Map] Merging narrative using target column: '{EXCEL_TEXT_COLUMN_NAME}'\n")

if 'fraud_reported' not in df.columns:
    raise KeyError("Column 'fraud_reported' is entirely missing from the primary CSV!")

df['fraud_reported'] = df['fraud_reported'].fillna('N')
target_dict = {'Y': 1, 'N': 0}
y = df['fraud_reported'].map(target_dict).values

cols_to_concat = ['incident_type', 'collision_type', 'incident_severity', 'insured_hobbies', EXCEL_TEXT_COLUMN_NAME]
for c in cols_to_concat:
    if c not in df.columns:
        df[c] = ""

df[cols_to_concat] = df[cols_to_concat].astype(str).fillna("")

df['Narrative'] = df['incident_type'] + " " + df['collision_type'] + " " + \
                  df['incident_severity'] + " " + df['insured_hobbies'] + " " + \
                  df[EXCEL_TEXT_COLUMN_NAME]

texts = df['Narrative'].values

np.random.seed(42)
indices = np.random.permutation(len(texts))
split_idx = int(0.8 * len(texts))
train_idx, val_idx = indices[:split_idx], indices[split_idx:]

X_text_train, X_text_val = texts[train_idx], texts[val_idx]
y_train, y_val = y[train_idx], y[val_idx]

In [ ]:
# ---------------------------------------------------------
# 2. The Scratch_TF_IDF Class
# ---------------------------------------------------------
class Scratch_TF_IDF:
    def __init__(self):
        self.vocab = {}
        self.idf = {}
        
    def fit(self, texts):
        df_counts = {}
        N = len(texts)
        for text in texts:
            words = set(str(text).lower().split())
            for w in words:
                df_counts[w] = df_counts.get(w, 0) + 1
        
        self.vocab = {w: i for i, w in enumerate(df_counts.keys())}
        
        for w, count in df_counts.items():
            self.idf[w] = math.log((1 + N) / (1 + count)) + 1
            
    def transform(self, texts):
        matrix = np.zeros((len(texts), len(self.vocab)))
        for i, text in enumerate(texts):
            words = str(text).lower().split()
            if len(words) == 0:
                continue
            
            tf_counts = {}
            for w in words:
                tf_counts[w] = tf_counts.get(w, 0) + 1
            
            for w, count in tf_counts.items():
                if w in self.vocab:
                    idx = self.vocab[w]
                    tf = count / len(words)
                    matrix[i, idx] = tf * self.idf[w]
                    
        return matrix

tfidf = Scratch_TF_IDF()
tfidf.fit(X_text_train)
X_train = tfidf.transform(X_text_train)
X_val = tfidf.transform(X_text_val)

In [ ]:
# ---------------------------------------------------------
# 3. The Scratch_Naive_Bayes Class
# ---------------------------------------------------------
try:
    import cupy as xp
    print("[Status]: CuPy detected! Offloading heavy matrix operations to T4 GPUs.")
except ImportError:
    import numpy as xp
    print("[Status]: CuPy not found. Falling back to CPU with pure NumPy.")

class Scratch_Naive_Bayes:
    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.classes_ = None
        self.class_log_prior_ = None
        self.feature_log_prob_ = None
        
    def fit(self, X, y):
        X = xp.array(X)
        y = xp.array(y)
        
        self.classes_ = xp.unique(y)
        n_classes = len(self.classes_)
        n_features = X.shape[1]
        
        self.class_log_prior_ = xp.zeros(n_classes)
        self.feature_log_prob_ = xp.zeros((n_classes, n_features))
        
        for idx, c in enumerate(self.classes_):
            X_c = X[y == c]
            self.class_log_prior_[idx] = math.log(X_c.shape[0] / X.shape[0])
            
            smoothed_counts = xp.sum(X_c, axis=0) + self.alpha
            self.feature_log_prob_[idx, :] = xp.log(smoothed_counts / xp.sum(smoothed_counts))
            
    def predict_proba(self, X):
        X = xp.array(X)
        log_probs = xp.zeros((X.shape[0], len(self.classes_)))
        for idx, c in enumerate(self.classes_):
            log_probs[:, idx] = self.class_log_prior_[idx] + X.dot(self.feature_log_prob_[idx, :])
            
        max_log_probs = xp.max(log_probs, axis=1, keepdims=True)
        exp_log_probs = xp.exp(log_probs - max_log_probs)
        res = exp_log_probs / xp.sum(exp_log_probs, axis=1, keepdims=True)
        if hasattr(res, 'get'):
            return res.get() 
        return res

    def predict(self, X):
        probs = self.predict_proba(X)
        clss = self.classes_.get() if hasattr(self.classes_, 'get') else self.classes_
        return clss[np.argmax(probs, axis=1)]

In [ ]:
# ---------------------------------------------------------
# 4. Analytics & Visualization (Overfitting & Optimization)
# ---------------------------------------------------------
def calc_log_loss(y_true, y_pred_proba):
    epsilon = 1e-15
    y_pred_proba = np.clip(y_pred_proba, epsilon, 1 - epsilon)
    loss = -np.mean(y_true * np.log(y_pred_proba) + (1 - y_true) * np.log(1 - y_pred_proba))
    return loss

def calc_f1(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    precision = tp / (tp + fp) if tp + fp > 0 else 0
    recall = tp / (tp + fn) if tp + fn > 0 else 0
    if precision + recall == 0: return 0
    return 2 * precision * recall / (precision + recall)

alphas_to_test = [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]
train_losses, val_losses, val_f1s = [], [], []

print("Tuning Alpha (Laplace Smoothing) to Avoid Overfitting:\n")
for a in alphas_to_test:
    model = Scratch_Naive_Bayes(alpha=a)
    model.fit(X_train, y_train)
    
    train_probs = model.predict_proba(X_train)[:, 1]
    val_probs = model.predict_proba(X_val)[:, 1]
    val_preds = model.predict(X_val)
    
    t_loss = calc_log_loss(y_train, train_probs)
    v_loss = calc_log_loss(y_val, val_probs)
    v_f1 = calc_f1(y_val, val_preds)
    
    train_losses.append(t_loss)
    val_losses.append(v_loss)
    val_f1s.append(v_f1)
    
    print(f"Alpha = {a:5.2f} | Train Loss: {t_loss:.4f} | Validation Loss: {v_loss:.4f} | Validation F1: {v_f1:.4f}")

best_idx = np.argmin(val_losses)
best_alpha = alphas_to_test[best_idx]
print(f"\n>>> Best Alpha selected: {best_alpha} (Validation Loss: {val_losses[best_idx]:.4f}) <<<")

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
str_alphas = [str(a) for a in alphas_to_test]

ax[0].plot(str_alphas, train_losses, marker='o', label='Training Loss')
ax[0].plot(str_alphas, val_losses, marker='s', label='Validation Loss')
ax[0].set_title('Loss vs Alpha (Overfitting Check)')
ax[0].set_xlabel('Laplace Smoothing Alpha')
ax[0].set_ylabel('Binary Cross-Entropy Loss')
ax[0].grid(True)
ax[0].legend()

ax[1].plot(str_alphas, val_f1s, marker='^', color='green', label='Validation F1-Score')
ax[1].set_title('F1-Score vs Alpha')
ax[1].set_xlabel('Laplace Smoothing Alpha')
ax[1].set_ylabel('F1-Score')
ax[1].grid(True)
ax[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------
# 5. Export & Automation (Using Best Model)
# ---------------------------------------------------------
import pickle

final_model = Scratch_Naive_Bayes(alpha=best_alpha)
final_model.fit(X_train, y_train)

X_all = tfidf.transform(texts)
narrative_deception_scores = final_model.predict_proba(X_all)[:, 1]

# Ensure internal variables are standard CPU-bound numpy arrays before Pickling 
# (otherwise the Pickle file breaks if loaded on a system without an Nvidia GPU/Cupy)
if hasattr(final_model.classes_, 'get'):
    final_model.classes_ = final_model.classes_.get()
    final_model.class_log_prior_ = final_model.class_log_prior_.get()
    final_model.feature_log_prob_ = final_model.feature_log_prob_.get()

# Export the raw feature dataset mapping
np.save('narrative_deception_scores.npy', narrative_deception_scores)
print("Automated Numpy save complete: 'narrative_deception_scores.npy'. ")

# Export the actual Brain/Weights of our Text algorithms
with open('nlp_engine_tfidf.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
with open('nlp_engine_model.pkl', 'wb') as f:
    pickle.dump(final_model, f)
print("Automated Pickle save complete: 'nlp_engine_tfidf.pkl' and 'nlp_engine_model.pkl'.")

display(FileLink('narrative_deception_scores.npy'))
display(FileLink('nlp_engine_tfidf.pkl'))
display(FileLink('nlp_engine_model.pkl'))